In [14]:
!pip install -q langchain langchain-groq langchain-community langgraph tavily-python

In [15]:
import os
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import HumanMessage, SystemMessage

In [27]:
os.environ["GROQ_API_KEY"] = "YOUR_API_KEY"
os.environ["TAVILY_API_KEY"] = "YOUR_API_KEY"
# Get free Tavily key from: app.tavily.com (free tier: 1000 searches/month)

print("API keys set!")

API keys set!


In [28]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.3
)

search_tool = TavilySearchResults(max_results=5)

print("LLM and search tool ready!")

LLM and search tool ready!


In [29]:
class ResearchState(TypedDict):
    topic: str
    search_results: str
    analysis: str
    final_report: str
    revision_needed: bool
    revision_feedback: str
    iteration: int

In [30]:
def research_agent(state: ResearchState) -> ResearchState:
    """Agent 1: Searches the web for information on the topic."""

    topic = state["topic"]
    iteration = state.get("iteration", 0)

    print(f"\n--- RESEARCH AGENT (Iteration {iteration + 1}) ---")
    print(f"Searching for: {topic}")

    # If revision needed, refine search based on feedback
    if state.get("revision_needed") and state.get("revision_feedback"):
        search_query = f"{topic} {state['revision_feedback']}"
        print(f"Refined search: {search_query}")
    else:
        search_query = topic

    # Search the web
    results = search_tool.invoke(search_query)

    # Format results
    formatted = ""
    for i, result in enumerate(results, 1):
        formatted += f"\nSource {i}: {result.get('url', 'N/A')}\n"
        formatted += f"Content: {result.get('content', 'N/A')}\n"
        formatted += "-" * 40

    print(f"Found {len(results)} sources")

    return {
        **state,
        "search_results": formatted,
        "iteration": iteration + 1
    }

In [31]:
def analysis_agent(state: ResearchState) -> ResearchState:
    """Agent 2: Analyzes the search results and extracts key insights."""

    print(f"\n--- ANALYSIS AGENT ---")
    print("Analyzing search results...")

    prompt = f"""You are a Research Analyst. Analyze the following search results
on the topic: "{state['topic']}"

Search Results:
{state['search_results']}

Provide a structured analysis with:
1. Key Findings (bullet points)
2. Important Statistics or Data
3. Different Perspectives or Viewpoints
4. Gaps in Information (what's missing)

Be thorough and factual. Cite sources where possible."""

    response = llm.invoke([HumanMessage(content=prompt)])
    analysis = response.content

    print("Analysis complete!")

    return {
        **state,
        "analysis": analysis
    }

In [32]:
def synthesis_agent(state: ResearchState) -> ResearchState:
    """Agent 3: Synthesizes analysis into a comprehensive final report."""

    print(f"\n--- SYNTHESIS AGENT ---")
    print("Writing final report...")

    prompt = f"""You are a Research Report Writer. Based on the following analysis,
write a comprehensive research report on: "{state['topic']}"

Analysis:
{state['analysis']}

Raw Search Results:
{state['search_results']}

Write a well-structured report with:
1. Executive Summary (2-3 sentences)
2. Introduction
3. Key Findings
4. Detailed Analysis
5. Conclusion
6. Sources

Make it professional, clear, and well-organized.
Include specific data and statistics where available."""

    response = llm.invoke([HumanMessage(content=prompt)])
    report = response.content

    print("Report written!")

    return {
        **state,
        "final_report": report
    }

In [33]:
def quality_check_agent(state: ResearchState) -> ResearchState:
    """Agent 4: Evaluates the report quality and decides if revision is needed."""

    print(f"\n--- QUALITY CHECK AGENT ---")
    print("Evaluating report quality...")

    prompt = f"""You are a Quality Reviewer. Evaluate this research report on: "{state['topic']}"

Report:
{state['final_report']}

Rate the report on these criteria (1-10):
1. Completeness: Does it cover the topic thoroughly?
2. Accuracy: Are claims supported by sources?
3. Structure: Is it well-organized?
4. Clarity: Is it easy to understand?

If the average score is below 7, respond with:
REVISION_NEEDED: YES
FEEDBACK: [specific improvements needed]

If the average score is 7 or above, respond with:
REVISION_NEEDED: NO
FEEDBACK: Report meets quality standards.

Also provide the scores."""

    response = llm.invoke([HumanMessage(content=prompt)])
    evaluation = response.content

    print(f"Evaluation:\n{evaluation[:300]}...")

    # Check if revision needed
    revision_needed = "REVISION_NEEDED: YES" in evaluation.upper()

    # Don't revise more than 2 times
    if state.get("iteration", 0) >= 3:
        revision_needed = False
        print("Max iterations reached. Finalizing report.")

    feedback = ""
    if "FEEDBACK:" in evaluation:
        feedback = evaluation.split("FEEDBACK:")[-1].strip()

    return {
        **state,
        "revision_needed": revision_needed,
        "revision_feedback": feedback
    }

In [34]:
def should_revise(state: ResearchState) -> str:
    """Decides whether to revise or finalize the report."""

    if state.get("revision_needed", False):
        print("\n>> Decision: REVISE — sending back to Research Agent")
        return "revise"
    else:
        print("\n>> Decision: FINALIZE — report is ready!")
        return "finalize"

In [35]:
# Create the workflow graph
workflow = StateGraph(ResearchState)

# Add nodes (agents)
workflow.add_node("researcher", research_agent)
workflow.add_node("analyzer", analysis_agent)
workflow.add_node("synthesizer", synthesis_agent)
workflow.add_node("quality_checker", quality_check_agent)

# Define edges (flow)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "analyzer")
workflow.add_edge("analyzer", "synthesizer")
workflow.add_edge("synthesizer", "quality_checker")

# Conditional edge — revise or finalize
workflow.add_conditional_edges(
    "quality_checker",
    should_revise,
    {
        "revise": "researcher",
        "finalize": END
    }
)

# Compile the graph
app = workflow.compile()

print("Multi-Agent Research System built successfully!")
print("\nAgent Flow:")
print("  Research Agent --> Analysis Agent --> Synthesis Agent --> Quality Check")
print("       ^                                                       |")
print("       |_________________ (if revision needed) _______________|")

Multi-Agent Research System built successfully!

Agent Flow:
  Research Agent --> Analysis Agent --> Synthesis Agent --> Quality Check
       ^                                                       |
       |_________________ (if revision needed) _______________|


In [36]:
def research(topic: str):
    """Run the multi-agent research system on a topic."""

    print("=" * 60)
    print(f"  MULTI-AGENT RESEARCH SYSTEM")
    print(f"  Topic: {topic}")
    print("=" * 60)

    initial_state = {
        "topic": topic,
        "search_results": "",
        "analysis": "",
        "final_report": "",
        "revision_needed": False,
        "revision_feedback": "",
        "iteration": 0
    }

    # Run the graph
    result = app.invoke(initial_state)

    print("\n" + "=" * 60)
    print("  FINAL REPORT")
    print("=" * 60)
    print(result["final_report"])
    print("=" * 60)
    print(f"\nTotal iterations: {result['iteration']}")

    return result

In [37]:
result = research("Impact of Artificial Intelligence on Healthcare in 2025")

  MULTI-AGENT RESEARCH SYSTEM
  Topic: Impact of Artificial Intelligence on Healthcare in 2025

--- RESEARCH AGENT (Iteration 1) ---
Searching for: Impact of Artificial Intelligence on Healthcare in 2025
Found 5 sources

--- ANALYSIS AGENT ---
Analyzing search results...
Analysis complete!

--- SYNTHESIS AGENT ---
Writing final report...
Report written!

--- QUALITY CHECK AGENT ---
Evaluating report quality...
Evaluation:
I have evaluated the research report on the "Impact of Artificial Intelligence on Healthcare in 2025" based on the given criteria. Here are my ratings:

1. Completeness: 8 - The report covers the topic thoroughly, discussing various aspects of AI in healthcare, including its impact on patient care, ...

>> Decision: FINALIZE — report is ready!

  FINAL REPORT
**Comprehensive Research Report: Impact of Artificial Intelligence on Healthcare in 2025**

**Executive Summary**
The integration of Artificial Intelligence (AI) in healthcare is expected to revolutionize the ind

In [38]:
result = research("Effects of climate change on global food security")

  MULTI-AGENT RESEARCH SYSTEM
  Topic: Effects of climate change on global food security

--- RESEARCH AGENT (Iteration 1) ---
Searching for: Effects of climate change on global food security
Found 5 sources

--- ANALYSIS AGENT ---
Analyzing search results...
Analysis complete!

--- SYNTHESIS AGENT ---
Writing final report...
Report written!

--- QUALITY CHECK AGENT ---
Evaluating report quality...
Evaluation:
**Scores:**
1. Completeness: 8
2. Accuracy: 9
3. Structure: 9
4. Clarity: 8

**Average Score:** 8.5

**REVISION_NEEDED:** NO
**FEEDBACK:** Report meets quality standards. The report provides a comprehensive overview of the effects of climate change on global food security, covering key findings, sta...

>> Decision: FINALIZE — report is ready!

  FINAL REPORT
**Comprehensive Research Report: Effects of Climate Change on Global Food Security**

**Executive Summary**
Climate change poses a significant threat to global food security, with rising temperatures and extreme weather even

In [39]:
result = research("Latest advancements in quantum computing 2025")

  MULTI-AGENT RESEARCH SYSTEM
  Topic: Latest advancements in quantum computing 2025

--- RESEARCH AGENT (Iteration 1) ---
Searching for: Latest advancements in quantum computing 2025
Found 5 sources

--- ANALYSIS AGENT ---
Analyzing search results...
Analysis complete!

--- SYNTHESIS AGENT ---
Writing final report...
Report written!

--- QUALITY CHECK AGENT ---
Evaluating report quality...
Evaluation:
I have evaluated the research report on "Latest advancements in quantum computing 2025" based on the given criteria. Here are my ratings:

1. Completeness: 8 - The report covers the topic thoroughly, providing an overview of the latest developments in quantum computing, including key findings, detai...

>> Decision: FINALIZE — report is ready!

  FINAL REPORT
**Executive Summary**
The latest advancements in quantum computing have made significant progress in 2025, with major breakthroughs in logical qubits, quantum error correction, and practical applications. Companies like IonQ, Google

In [40]:
print("=" * 50)
print("MULTI-AGENT RESEARCH SYSTEM — PROJECT STATS")
print("=" * 50)
print(f"Framework:          LangGraph")
print(f"LLM:                Groq (Llama 3.3 70B)")
print(f"Search Tool:        Tavily API")
print(f"Agents:             4 (Research, Analysis, Synthesis, Quality)")
print(f"Flow:               Cyclic graph with conditional revision")
print(f"Max Iterations:     3")
print(f"Self-Correction:    Yes (Quality Agent triggers revision)")
print("=" * 50)

MULTI-AGENT RESEARCH SYSTEM — PROJECT STATS
Framework:          LangGraph
LLM:                Groq (Llama 3.3 70B)
Search Tool:        Tavily API
Agents:             4 (Research, Analysis, Synthesis, Quality)
Flow:               Cyclic graph with conditional revision
Max Iterations:     3
Self-Correction:    Yes (Quality Agent triggers revision)
